In [8]:
import os
import tempfile
import shutil
from pathlib import Path
from lxml import etree
from lxml.etree import ElementTree, Element

# DIR_DATA_SAMPLE = Path(Path.cwd().parent, "data")
# DIR_SEMPER_SAMPLE = Path(DIR_DATA_SAMPLE, "semper")
# DIR_HWGW_SAMPLE = Path(DIR_DATA_SAMPLE, "hwgw")
DIR_OUT = Path(Path.cwd().parent, "_dts_out")

# list(DIR_SEMPER_SAMPLE.glob("*.*"))
# list(DIR_HWGW_SAMPLE.glob("*.*"))

import logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)


NS = {"tei": "http://www.tei-c.org/ns/1.0"}

In [ ]:
from copy import deepcopy


def parse_xml(path: Path | str) -> ElementTree:
    parser = etree.XMLParser(remove_blank_text=True, load_dtd=True)
    return etree.parse(str(path), parser)


def print_etree(etre: ElementTree):
    print(etree.tostring(etre, pretty_print=True, encoding="unicode"))


def write_xml_with_schema(tree: ElementTree, output_path: Path) -> None:
    root = deepcopy(tree.getroot())
    pi = etree.ProcessingInstruction("oxygen", 'RNGSchema="schema.rng" type="xml"')
    root.addprevious(pi)
    output_path.parent.mkdir(exist_ok=True, parents=True)
    # output_path.mkdir(exist_ok=True, parents=True)
    etree.ElementTree(root).write(
        output_path, pretty_print=True, xml_declaration=True, encoding="UTF-8"
    )

# II. Recursive Parsing

In [4]:
DIR_SAMPLE = Path.cwd().parent / "_dev" / "test-data_dts"
base_folder = DIR_SAMPLE
base_identifier = "http://example/org/dts/semper/semper-edition"

In [24]:
NS = {"tei": "http://www.tei-c.org/ns/1.0"}

REFERENCES_DECLARATION = [
    # Logical: paragraph > lines
    etree.XML("""
        <refsDecl xmlns="http://www.tei-c.org/ns/1.0"
                  default="true"
                  n="logical_structure">
            <citeStructure unit="paragraph" match="//p" use="@xml:id">
                <citeStructure unit="line" match=".//lb" use="@n" delim="-"/>
            </citeStructure>
        </refsDecl>
    """),
    # Pages
    etree.XML("""
        <refsDecl xmlns="http://www.tei-c.org/ns/1.0"
                  n="published_page">
            <citeStructure unit="page" match="//pb" use="@xml:id" />
        </refsDecl>
    """)
]

def inject_citeStructure_in_refsDecl(
    tree: ElementTree, 
    references_declaration: list
) -> ElementTree:
    existing = tree.xpath("//tei:refsDecl", namespaces=NS)
    if existing:
        logger.debug("   refsDecl already declared in TEI Header - skipping")
        return tree

    root = deepcopy(tree.getroot())
    new_tree = etree.ElementTree(root)

    tei_encDesc = tree.xpath(
        "/tei:TEI/tei:teiHeader/tei:encodingDesc", namespaces=NS
    )
    # If no `encodingDesc` found, create and fill it with the citeStructure
    if not tei_encDesc:
        # First check if it is really a TEI File
        tei_header_list = new_tree.xpath(
            "/tei:TEI/tei:teiHeader", namespaces=NS
        )
        if not tei_header_list:
            raise ValueError("No teiHeader found, valid XML TEI ?")
        
        # Then log what we are doing (creating encodingDesc)
        tei_title_nodes = tree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            namespaces=NS,
        )
        tree_tei_title = (
            "".join(tei_title_nodes[0].itertext())
            if tei_title_nodes
            else "title not found"
        )
        logger.debug(
            f"  No encodingDesc found in TEI Header for '{tree_tei_title}', creating the header to make it DTS-Searchable"
        )
        
        # And finally create it
        new_encDesc = etree.SubElement(tei_header_list[0], "{http://www.tei-c.org/ns/1.0}encodingDesc") # Using Clark Notation for namespacing
        tei_encDesc = new_tree.xpath(
            "/tei:TEI/tei:teiHeader/tei:encodingDesc", namespaces=NS
        )
        
    
    tei_encDesc = tei_encDesc[0]
    for ref in references_declaration:
        # print(f'encDesc: {etree.tostring(tei_encDesc, pretty_print=True, encoding="unicode")}')
        tei_encDesc.append(deepcopy(ref)) # lxml moves element before appending? so deepcopy to not working by reference

    return new_tree


In [25]:
def process_tei_folder_with_facsimile_leafs(
    tei_folder: Path,
    base_identifier: str,
    destination_folder: Path,
    base_relative_path_to_destination_tei_folder: Path | str = "../../tei/semper-edition",
    base_tei_folder: Path | None = None,
    current_collection_identifier: str | None = None,
    current_catalog: Element | None = None,
): # -> Tuple[lxml.Element, Path]
    xml_files = sorted(tei_folder.glob("*.xml"))
    tei_subfolders = [f for f in tei_folder.iterdir() if f.is_dir()]

    manuscript_identifier = f"{base_identifier}/{tei_folder.name}"

    # -- Build the manuscript catalog at first recurrence --
    if current_catalog is None:
        # Create the manuscript catalog base collection 
        logger.info(f"Building Manuscript Catalog for {tei_folder.name}")
        manuscript_catalog_element = etree.Element(
            "collection",
            identifier=manuscript_identifier,
        )
        # ... with metadata from the first .xml encountered
        all_xml_files = sorted(tei_folder.glob("**/*.xml"))
        if all_xml_files:
            title_elem = etree.SubElement(manuscript_catalog_element, "title")
            first_tei_tree = parse_xml(all_xml_files[0])
            extracted_titles = first_tei_tree.xpath(
                "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
                namespaces=NS,
            )
            title_elem.text = (
                extracted_titles[0].text.strip() if extracted_titles else tei_folder.name
            )
        # Initialize the root recurrence
        current_catalog = deepcopy(manuscript_catalog_element)                  # the catalog being worked on
        current_collection_identifier = deepcopy(manuscript_identifier)         # the current identifier of the collection
        base_tei_folder=tei_folder


    # -- Find the current collection element in the current catalog --
    match_current_collection = current_catalog.xpath(
        f"descendant-or-self::*[@identifier='{current_collection_identifier}']"
    )
    current_collection = match_current_collection[0]
    current_collection_members = etree.SubElement(current_collection, "members")

    current_tei_relative_directory = tei_folder.relative_to(base_tei_folder.parent)

    # -- Add .xml files as collection resource and copy the TEI file  --
    for xml_file in xml_files:
        resource_xml_file_relative_path = f"{base_relative_path_to_destination_tei_folder}/{current_tei_relative_directory}/{xml_file.name}"

        # ---- Catalog and Collections ----
        # Add the file as a resource to the current collection depth
        xml_file_etree = parse_xml(xml_file)
        resource_identifier = f"{current_collection_identifier}/{xml_file.stem}"
        resource_element = etree.SubElement(
            current_collection_members,
            "resource",
            identifier=resource_identifier,
            filepath=resource_xml_file_relative_path
        )
        # ... with metadata from the xml_file
        resource_title = etree.SubElement(resource_element, "title")
        tei_title_nodes = xml_file_etree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            namespaces=NS,
        )
        resource_title.text = (
            "".join(tei_title_nodes[0].itertext())
            if tei_title_nodes
            else tei_folder.stem
        )

        # ---- TEI Files and TemporaryDirectory  -----
        # ------ (part.1) Copy .xml -------
        xml_file_etree_naviguable = inject_citeStructure_in_refsDecl(
            tree=xml_file_etree, references_declaration=REFERENCES_DECLARATION
        )

        write_xml_with_schema(xml_file_etree_naviguable, Path(destination_folder, current_tei_relative_directory, xml_file.name))

    # ---- (part.2) Copy schema dependencies (.dtd, .rng) -----
    for f in tei_folder.glob("*"):
        if f.is_file() and f.suffix in {".dtd", ".rng"}:
            dst = Path(destination_folder, current_tei_relative_directory, f.name)
            dst.parent.mkdir(exist_ok=True, parents=True)
            shutil.copy(src=f, dst=dst)
            logger.debug("  copied schema file %s → %s", f.name, dst)



    # -- RECURSE, drill into subfolders --
    # -- Prepare first, adding the subfolders as new collections  --
    for subf in tei_subfolders:
        # Create the subfolder collectioin base collection 
        subfolder_identifier = f"{current_collection_identifier}/{subf.name}"
        subcollection_elem = etree.SubElement(
            current_collection_members,
            "collection",
            identifier=subfolder_identifier,
        )
        # ... with metadata from the first .xml encountered
        suball_xml_files = sorted(subf.glob("**/*.xml"))
        if suball_xml_files:
            subf_title_elem = etree.SubElement(subcollection_elem, "title")
            subf_tei_tree = parse_xml(suball_xml_files[0])
            subf_titles = subf_tei_tree.xpath(
                "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
                namespaces=NS,
            )
            subf_title_elem.text = (
                subf_titles[0].text.strip() if subf_titles else subf.name
            )
        # Recurse to populate this sub-collection's members
        process_tei_folder_with_facsimile_leafs(
            tei_folder=subf,
            base_identifier=base_identifier,
            base_tei_folder=base_tei_folder,
            current_collection_identifier=subfolder_identifier,
            current_catalog=current_catalog,
            destination_folder=destination_folder
        )

    return current_catalog, 0

# --- Discover TEI folders at `source` ---
base_tei_subfolders = sorted(p for p in base_folder.iterdir() if p.is_dir())
non_dirs = [p for p in base_folder.iterdir() if not p.is_dir()]
for p in non_dirs:
    logger.warning(f"Skipping non-directory entry in source: {p.name}")

if not base_tei_subfolders:
    logger.error(f"No sub-directories found in source: {base_tei_subfolders}")
    # sys.exit(1)

logger.info(f"Found {len(base_tei_subfolders)} TEI folder(s) to process in '{base_folder}'")

# --- Process each folder ---
processed, failed = 0, 0
tmpdir = tempfile.TemporaryDirectory(dir=Path.cwd())
logger.info(Path(tmpdir.name))
for tei_folder in base_tei_subfolders:
    logger.info(f"Processing folder: {tei_folder.stem}")
    ms_catalog, ms_tei_directory = process_tei_folder_with_facsimile_leafs(
        tei_folder=tei_folder,
        base_identifier=base_identifier,
        destination_folder=Path(tmpdir.name)
    )
    # logger.debug(etree.tostring(ms_catalog, pretty_print=True, encoding="unicode"))

INFO:__main__:Found 3 TEI folder(s) to process in '/Users/angros/Etudes/21b_Services-dts_dapytains/_dev/test-data_dts'
INFO:__main__:/Users/angros/Etudes/21b_Services-dts_dapytains/notebooks/tmp6kffa23v
INFO:__main__:Processing folder: 184
INFO:__main__:Building Manuscript Catalog for 184
DEBUG:__main__:  copied schema file semper_edition_schema_prov.rng → /Users/angros/Etudes/21b_Services-dts_dapytains/notebooks/tmp6kffa23v/184/semper_edition_schema_prov.rng
DEBUG:__main__:  copied schema file Zeichen.dtd → /Users/angros/Etudes/21b_Services-dts_dapytains/notebooks/tmp6kffa23v/184/Zeichen.dtd
INFO:__main__:Processing folder: 185
INFO:__main__:Building Manuscript Catalog for 185
DEBUG:__main__:  copied schema file semper_edition_schema_prov.rng → /Users/angros/Etudes/21b_Services-dts_dapytains/notebooks/tmp6kffa23v/185/semper_edition_schema_prov.rng
DEBUG:__main__:  copied schema file Zeichen.dtd → /Users/angros/Etudes/21b_Services-dts_dapytains/notebooks/tmp6kffa23v/185/Zeichen.dtd
INF

In [76]:
current_tempdir

NameError: name 'current_tempdir' is not defined

## I. Parsing

First, set the static collection metadata as an `lxml.ElementTree`

In [ ]:
catalog_header_semper: str = """<?xml version='1.0' encoding='UTF-8'?>
<?oxygen RNGSchema="schema.rng" type="xml"?>
<collection identifier="https://example.org/dts/collections/semper">
    <title>Digital Semper Edition (dev: collection)</title>
    <dublinCore>
        <creator>Gottfried Semper</creator>
        <language>de</language>
    </dublinCore>
    <!-- Add members -->
</collection>
"""

catalog_semper: ElementTree = etree.ElementTree(
    etree.fromstring(catalog_header_semper.encode("utf-8"))
)
print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding="unicode"))

In [ ]:
def parse_teifolder_to_dapytains(
    source: Path, target: Path, catalog: ElementTree
) -> etree.Element:
    def extract_volume_titles(etree):
        volume = etree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title",
            namespaces=NS,
        )

        monograph = etree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title[@level='m']",
            namespaces=NS,
        )

        titles = []
        if volume:
            titles.append(volume[0].text.strip())
        if monograph:
            titles.append(monograph[0].text.strip())
        return titles

    # Flow control
    if not source.exists():
        raise ValueError(f"Source directory does not exist: {source}")
    if not source.is_dir():
        raise ValueError(f"Source path is not a directory: {source}")

    ## Get the root node to retrieve the collection base_identifier
    catalog_root = catalog.getroot()
    base_identifier: str = catalog_root.get("identifier")
    if not base_identifier:
        raise ValueError("Catalog does not have an identifier attribute")

    ## (1) ROOT CATALOG - Make reference to Collection file in the catalog as members
    collection_xml_relative_path = f"./collection-{source.name}.xml"
    collection_catalog = etree.Element(
        "collection",
        identifier=f"{base_identifier}/{source.name}",
        filepath=str(collection_xml_relative_path),
    )

    ## (2) NEW 'Catalog - Manuscript as a Collection'
    collection_manuscript = etree.Element(
        "collection", identifier=f"{base_identifier}/{source.name}"
    )
    title = etree.SubElement(collection_manuscript, "title")
    first_file_etree = parse_xml(next(source.glob("*.xml")))
    extracted_title = first_file_etree.xpath(
        "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title", namespaces=NS
    )
    title.text = extracted_title[0].text.strip()
    members_collection_manuscript = etree.SubElement(collection_manuscript, "members")

    for source_tei_file in sorted(source.glob("*.xml")):
        source_tei_tree = parse_xml(source_tei_file)
        print(extract_volume_titles(etree=source_tei_tree))
        resource_identifier = f"{base_identifier}/{source.name}/{source_tei_file.stem}"
        print(resource_identifier)

        resource_xml_path = Path(target, "tei", source.name, source_tei_file.name)
        resource_xml_relative_path = f"../tei/{source.name}/{source_tei_file.name}"
        resource = etree.SubElement(
            members_collection_manuscript,
            "resource",
            identifier=resource_identifier,
            filepath=resource_xml_relative_path,
        )
        title = etree.SubElement(resource, "title")
        extracted_source_tei_title = source_tei_tree.xpath(
            "/tei:TEI/tei:teiHeader/tei:fileDesc/tei:titleStmt/tei:title", namespaces=NS
        )[0]
        title.text = "".join(extracted_source_tei_title.itertext())

        write_xml_with_schema(source_tei_tree, resource_xml_path)

    ## (2.2) Write the 'Catalog - Manuscript as a Collection'
    collection_target_path = Path(target, "catalog", f"collection-{source.name}.xml")
    collection_manuscript_tree = etree.ElementTree(collection_manuscript)
    write_xml_with_schema(collection_manuscript_tree, collection_target_path)

    ## (3) Copy local dependencies (.dtd, .rng)
    for f in source.glob("*"):
        if f.is_file() and f.suffix in [".dtd", ".rng"]:
            shutil.copy(src=f, dst=Path(target, "tei", source.name, f.name))

    return collection_catalog

In [ ]:
def append_collection_as_member_to_catalog(
    catalog_tree: ElementTree, collection: Element
) -> ElementTree:
    root = catalog_tree.getroot()
    members_element = root.find("members")
    if members_element is None:
        members_element = etree.SubElement(root, "members")
    members_element.append(collection)
    return catalog_tree

In [ ]:
for source_dir in sorted(DIR_SEMPER_SAMPLE.iterdir()):
    if not source_dir.is_dir():
        continue
    print(f"Processing {source_dir.name}...")
    cm = parse_teifolder_to_dapytains(
        source=source_dir, catalog=catalog_semper, target=DIR_OUT
    )
    catalog_semper = append_collection_as_member_to_catalog(
        catalog_tree=catalog_semper, collection=cm
    )

print("\nFinal catalog:")
print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding="unicode"))
write_xml_with_schema(
    tree=catalog_semper, output_path=Path(DIR_OUT, "catalog", "catalog.xml")
)

# WIPs

In [ ]:
# def process_tei_folder_with_facsimile_leafs(
#     tei_folder: Path,
#     base_identifier: str
# ): # -> Tuple[lxml.Element, Path]
#     xml_files = sorted(tei_folder.glob("**/*.xml"))
#     if not xml_files:
#         raise ValueError(f"No XML files found in directory: {tei_folder}")
    
#     # -- Build the manuscript catalog --
#     manuscript_identifier = f"{base_identifier}/{tei_folder.name}"
#     manuscript_catalog = etree.Element(
#         "collection",
#         identifier=manuscript_identifier,
#     )

#     # Brute and inelegant, shall be made recursive. 
#     for xml_file in xml_files:
#         relative_xml_filepath = xml_file.relative_to(tei_folder.parent)
#         subpath_list = sorted(list(relative_xml_filepath.parents), reverse=False)[1:]
        
#         # Add the subcollections if needed
#         for subp in subpath_list:
#             subp_current_identifier = f"{base_identifier}/{subp}"
#             if manuscript_catalog.xpath(f"{subp_current_identifier}") is None:
#                 print(subp)
#             print(subp_current_identifier)
        
#         # Add the xml_file as resource

In [ ]:
# def parse_flat_teifolder_to_dapytains(
#     source: Path,
#     target: Path,
#     catalog: ElementTree,
#     etree: Element | None = None,
# ):  # -> etree.Element:
#     # Flow control
#     if not source.exists():
#         raise ValueError(f"Source directory does not exist: {source}")
#     if not source.is_dir():
#         raise ValueError(f"Source path is not a directory: {source}")
#     if not target.exists():
#         raise ValueError(f"Target directory does not exist: {target}")
#     if not target.is_dir():
#         raise ValueError(f"Target path is not a directory: {target}")
#     # if not (("catalog" in [stem for stem in list(target.glob("*"))]) and ("tei" in [stem for stem in list(target.glob("*"))])):
#     #     raise ValueError(f"Init target folder with Dapytain' catalog, tei and catalog.xml: {list(target.glob("*"))}")
#     # if not list(source.glob("*.xml")): raise ValueError(f"No XML files found in directory: {source}")

#     # Get the root node, "An ElementTree is mainly a document wrapper around a tree with a root node."
#     if isinstance(catalog, ElementTree):
#         root = catalog.getroot()
#     else:
#         root = catalog  # avoid problems with _Element and so on

#     # ... to retrieve the base_identifier
#     base_identifier: str = root.get("identifier")
#     if not base_identifier:
#         raise ValueError("Catalog does not have an identifier attribute")

#     # (1) generate the member statement for the collection catalog as a new collection
#     collection_target_path = Path(target, "catalog", f"{source.name}.xml")
#     collection = etree.Element(
#         "collection", identifier=f"{base_identifier}/{source.name}"
#     )

#     # ... and its elements
#     title = etree.SubElement(collection, "title", filepath=collection_target_path)
#     title.text = f"Manuscript {source.name}"

#     # (2) write out the new collection at target path

#     return collection


# cm = parse_flat_teifolder_to_dapytains(
#     source=DIR_SEMPER_SAMPLE / "184", target=DIR_OUT, catalog=catalog_semper
# )
# print_etree(cm)

In [ ]:
# for source_dir in sorted(DIR_SEMPER_SAMPLE.iterdir()):
#     if not source_dir.is_dir():
#         continue
#     print(f"Processing {source_dir.name}...")
#     cm = parse_teifolder_to_dapytains(source=source_dir, catalog=catalog_semper, target=DIR_OUT)
#     catalog_semper = append_collection_as_member_to_catalog(catalog_tree=catalog_semper, collection=cm)

# print("\nFinal catalog:")
# print(etree.tostring(catalog_semper.getroot(), pretty_print=True, encoding='unicode'))
# write_xml_with_schema(tree=catalog_semper, output_path=Path(DIR_OUT, "catalog", "catalog.xml"))

In [ ]:
# def create_root_collection_catalog_hwgw(
#     root_dir: Path, root_tei_file: Path, base_identifier
# ) -> ElementTree:
#     """Generate the root collection catalog for a TEI File Folder"""
#     first_tei = parse_xml(root_tei_file)
#     collection_root = etree.Element("collection", identifier=base_identifier)

#     # Add the collection metadata to the catalog
#     title = etree.SubElement(collection_root, "title")
#     title.text = extract_root_series_title(first_tei)
#     dublin = etree.SubElement(collection_root, "dublinCore")
#     creator = etree.SubElement(dublin, "creator")
#     creator.text = "Heinrich Wölfflin"
#     language = etree.SubElement(dublin, "language")
#     language.text = "de"
#     members = etree.SubElement(collection_root, "members")

#     for folder in sorted(root_dir.iterdir()):
#         if folder.is_dir() and folder.name.startswith("s"):
#             etree.SubElement(members, "collection", filepath=f"./{folder.name}.xml")

#     return etree.ElementTree(collection_root)


# def create_root_collection_catalog_semper(
#     root_dir: Path, root_tei_file: Path, base_identifier
# ) -> ElementTree:
#     """Generate the root collection catalog for a TEI File Folder"""
#     first_tei = parse_xml(root_tei_file)
#     collection_root = etree.Element("collection", identifier=base_identifier)

#     # Add the collection metadata to the catalog
#     title = etree.SubElement(collection_root, "title")
#     title.text = extract_root_series_title(first_tei)
#     dublin = etree.SubElement(collection_root, "dublinCore")
#     creator = etree.SubElement(dublin, "creator")
#     creator.text = "Gottfried Semper"
#     language = etree.SubElement(dublin, "language")
#     language.text = "de"
#     members = etree.SubElement(collection_root, "members")

#     # # for folder in sorted(root_dir.iterdir()):
#     # #     if folder.is_dir() and folder.name.startswith("s"):
#     # #         etree.SubElement(
#     # #             members,
#     # #             "collection",
#     # #             filepath=f"./{folder.name}.xml"
#     # #         )

#     return etree.ElementTree(collection_root)


# # catalog_hwgw = create_root_collection_catalog_hwgw(
# #     root_dir=DIR_HWGW_SAMPLE,
# #     root_tei_file=root_hwgw_tei,
# #     base_identifier='Unk'
# # )

In [ ]:
# root_catalog_semper = """<?xml version='1.0' encoding='UTF-8'?>
# <?oxygen RNGSchema="schema.rng" type="xml"?>
# <collection identifier="https://example.org/dts/collections/semper">
#     <title>Collection Semper</title>
#     <dublinCore>
#         <creator>Gottfried Semper</creator>
#         <language>de</language>
#     </dublinCore>
#     <!-- Add members -->
# </collection>
# """


# def create_subcollection_from_flat_dir() -> ElementTree:
#     return etree.ElementTree(subcollection)


# def create_collection(base_identifier, source_dir, metadatas: str) -> ElementTree:
#     collection = etree.Element("collection", identifier=base_identifier)
#     collection_members = etree.SubElement(collection, "members")
#     for folder in sorted(source_dir.iterdir()):
#         if not folder.is_dir():
#             # print(f"Exists file {folder}")
#             continue

#         etree.SubElement(
#             collection_members, "collection", filepath=f"./{folder.name}.xml"
#         )
#         for f in sorted(folder.iterdir()):
#             if not f.suffix == ".xml":
#                 # print(f"Skipping - {f}")
#                 continue

#     return etree.ElementTree(collection)


# c = create_collection_semper(
#     base_identifier="semper-digital-editions",
#     source_dir=DIR_SEMPER_SAMPLE,
#     metadatas=root_catalog_semper,
# )

In [ ]:
# c.tostring()